# 04 – Tap og læring uten å utlede backpropagation

**Tid:** 45 minutter. **Mål:** skille tap fra nøyaktighet og forklare hva de
fem treningsstegene gjør. Vi utleder verken derivasjon eller backpropagation.
Filnavnet er beholdt for at eksisterende lenker skal virke.

PyTorch beregner gradientene for oss.
En gradient beskriver lokal endring i tap når en parameter endres.
Optimizer bruker dette til å foreslå nye parametere.


In [ ]:
import torch
from torch import nn


## Eksempel: samme prediksjon, forskjellig tap

`CrossEntropyLoss` får rå skårer `[B, K]` og heltallsetiketter `[B]`.
Tapet er ett desimaltall; lavere er bedre etter tapsfunksjonens mål.
Vi bruker ikke softmax først. Nøyaktighet ser bare hvilken skår som er høyest.


In [ ]:
loss_fn = nn.CrossEntropyLoss()
targets = torch.tensor([0, 2])
weak_scores = torch.tensor([[0.1, 0., 0.], [0., 0., 0.1]])
strong_scores = torch.tensor([[4., 0., 0.], [0., 0., 4.]])
for name, scores in [("Svake", weak_scores), ("Sterke", strong_scores)]:
    accuracy = (scores.argmax(dim=1) == targets).float().mean().item()
    print(name, "tap:", loss_fn(scores, targets).item(), "nøyaktighet:", accuracy)


## Oppgave 1: Riktig, usikker og selvsikkert feil

Bruk etikettene `[2, 7]` og to rader med ti skårer.
Beregn `equal_loss` for bare nuller; `better_loss` når riktig klasse har skår 4;
og `wrong_loss` når klasse 0 har skår 4 i begge rader og resten er 0.
Skriv ut tapene og nøyaktigheten i hvert tilfelle. Forutsi rangeringen først.

<details>
<summary>Hint</summary>

Lag tre tabeller med `torch.zeros(2, 10)` og `clone()`. Klassen er kolonneindeksen.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør etter at du har løst oppgaven. Undersøk verdier og former hvis en kontroll feiler.


In [ ]:
assert better_loss < equal_loss < wrong_loss
assert labels.shape == (2,) and labels.dtype == torch.int64
assert torch.allclose(equal_loss, torch.tensor(10.).log())


### Refleksjon

Hvorfor kan nøyaktighet være lik selv om tapene er forskjellige?
Hvorfor bruker vi ikke avstanden mellom predikert siffer og riktig siffer som tap?


**Mitt svar:**

Skriv din forklaring her.


## Eksempel: fem steg, to forskjellige handlinger

Les linjene høyt: nullstill → prediker → mål tap → beregn gradienter → oppdater.
`backward()` beregner gradienter med backpropagation, men endrer ikke vektene.
`step()` oppdaterer dem. Gradienter samles opp i PyTorch, så `zero_grad()`
nullstiller før en ny uavhengig oppdatering. Du skal kjenne rollene; du trenger
ikke kunne algoritmen bak `backward()`.


In [ ]:
x = torch.tensor([[1., 0.], [0., 1.]])
y = torch.tensor([0, 1])
example_model = nn.Linear(2, 2)
example_optimizer = torch.optim.SGD(example_model.parameters(), lr=0.1)
example_optimizer.zero_grad()
scores = example_model(x)
loss = loss_fn(scores, y)
loss.backward()
example_optimizer.step()
print("Tap før oppdateringen:", loss.item())


## Oppgave 2: Se når vektene faktisk endres

Lag `model = nn.Linear(2, 2)` og en ny SGD-optimizer med læringsrate 0.1.
Bruk `x` og `y` fra eksemplet. Lag `weights_before = model.weight.detach().clone()`.
Skriv de fem treningsstegene. Rett etter `backward()` (før `step()`),
lag `weights_after_backward` som en kopi og `gradient` fra `model.weight.grad.clone()`.
Etter `step()` lag `weights_after_step` som en kopi.
Skriv ut de tre vektmatrisene og gradienten.

<details>
<summary>Hint</summary>

Bruk `detach().clone()` som et øyeblikksbilde. Sørg for at kopien etter `backward()` tas før `step()`.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør etter at du har løst oppgaven. Undersøk verdier og former hvis en kontroll feiler.


In [ ]:
assert torch.equal(weights_before, weights_after_backward)
assert not torch.equal(weights_before, weights_after_step)
assert torch.allclose(weights_after_step, weights_before - 0.1 * gradient)


### Refleksjon

Hvilken linje endret vektene? Hvorfor tar vi en kopi ved sammenligning?
Hva skjer hvis vi hopper over `step()`? Hva er læringsratens rolle?


**Mitt svar:**

Skriv din forklaring her.


## Oppgave 3: Finn feilene i treningsoppskriften

En kollega foreslår denne rekkefølgen (les den, ikke kjør den):

```python
scores = model(x).softmax(dim=1)
loss = loss_fn(scores, y.float())
optimizer.step()
loss.backward()
```

Skriv én korrekt treningsrunde med modellen og optimizer fra oppgave 2.
Forklar minst fire feil i forslaget i svarcellen.

<details>
<summary>Hint</summary>

Kontroller skårer, etikettens datatype, nullstilling og rekkefølgen på de to siste handlingene.

</details>


In [ ]:
# Skriv din kode her.


**Kontroll:** Kjør etter at du har løst oppgaven. Undersøk verdier og former hvis en kontroll feiler.


In [ ]:
assert scores.shape == (2, 2) and y.dtype == torch.int64


### Refleksjon

Hvorfor betyr rekkefølgen noe? Når kan softmax brukes?


**Mitt svar:**

Skriv din forklaring her.


## Valgfri demonstrasjon: læringsrate

Kjør samme lille problem fra samme start med tre læringsrater. Dette er en
illustrasjon, ingen utledning. Oppgaven i MNIST blir å observere om tapet
faller over flere epoker, ikke å garantere at hvert steg alltid gir lavere tap.


In [ ]:
for learning_rate in [0.01, 0.1, 1.1]:
    parameter = nn.Parameter(torch.tensor(0.0))
    optimizer = torch.optim.SGD([parameter], lr=learning_rate)
    for _ in range(20):
        optimizer.zero_grad()
        loss = (parameter - 5) ** 2
        loss.backward()
        optimizer.step()
    print(f"Læringsrate {learning_rate}: sluttverdi {parameter.item():.3f}, "
          f"sluttap {((parameter - 5) ** 2).item():.4f}")


**Neste steg:** Samme fem steg skal nå kjøres for grupper med MNIST-bilder.


## Videre lesing

Lenkene er på engelsk og er valgfrie oppslag eller fordypning. Du trenger ikke lese alt for å løse oppgavene.

- **Start her:** [PyTorch – Optimizing Model Parameters](https://docs.pytorch.org/tutorials/beginner/basics/optimization_tutorial.html). Les delene om tapsfunksjon og optimizer. Finn igjen `zero_grad()`, `backward()` og `step()`, og forklar forskjellen mellom de to siste.
- **Valgfri fordypning etter kurset:** [PyTorch – Automatic Differentiation with torch.autograd](https://docs.pytorch.org/tutorials/beginner/basics/autogradqs_tutorial.html). Forklarer gradientberegningen bak `backward()`. Beregningsgrafer og matematiske detaljer er ikke forkunnskaper eller læringskrav i dette kurset.
